# Multi-Agent · Day 31 — **Multi-Agent Architecture Fundamentals**

**Phase 2 · Module 5: Multi-Agent Orchestration** · ~2.5 hrs

Phase 1 built *one* good agent. Phase 2 asks the harder question: when should there be more than one, and how do they talk? The Barclays JD names multi-agent orchestration explicitly, and the interview question is never "can you install CrewAI" — it's *"you have loan processing end-to-end; how many agents, in what topology, and why?"*

Today:
1. The three **topologies** — supervisor, hierarchical, peer-to-peer.
2. **Message passing vs shared state** — the real architectural fork.
3. A runnable **shared-state** graph vs a **message-passing** crew, same task.
4. **Barclays mappings**: loan processing (supervisor) and fraud detection (pipeline + peers).
5. The **cost** of a handoff — why more agents is often worse.

Everything runs keyless — the "LLM" is a deterministic stub so the *topology* is what you study.

## 1. The three topologies

Every multi-agent system is one of these three, or a composition of them. Print them side by side; you'll redraw this diagram in an interview.

In [1]:
TOPOLOGIES = [
    ("Supervisor",   "1 router -> N specialists, all replies return to the router",
     "clear audit trail, easy fallback", "router is a bottleneck + single point of failure",
     "loan processing, customer service triage"),
    ("Hierarchical", "supervisor of supervisors; each team has its own router",
     "scales past ~8 agents, team-local context", "deep chains = latency + diluted context",
     "whole KYC division: onboarding team + screening team"),
    ("Peer-to-peer", "agents hand off directly to each other, no central router",
     "fewest hops, flexible", "no global view, can loop forever, hard to audit",
     "research/debate, code-review pairs"),
]

for name, shape, pro, con, use in TOPOLOGIES:
    print(f"{name.upper()}")
    print(f"  shape : {shape}")
    print(f"  pro   : {pro}")
    print(f"  con   : {con}")
    print(f"  use   : {use}\n")

print(r"""
  SUPERVISOR              HIERARCHICAL                 PEER-TO-PEER

      [S]                     [Top]                      [A]---[B]
    /  |  \                  /     \                      | \ / |
  [A] [B] [C]            [S1]       [S2]                  |  X  |
   \   |   /             /  \       /  \                  | / \ |
      [S]             [A]  [B]   [C]  [D]                [C]---[D]
""")

SUPERVISOR
  shape : 1 router -> N specialists, all replies return to the router
  pro   : clear audit trail, easy fallback
  con   : router is a bottleneck + single point of failure
  use   : loan processing, customer service triage

HIERARCHICAL
  shape : supervisor of supervisors; each team has its own router
  pro   : scales past ~8 agents, team-local context
  con   : deep chains = latency + diluted context
  use   : whole KYC division: onboarding team + screening team

PEER-TO-PEER
  shape : agents hand off directly to each other, no central router
  pro   : fewest hops, flexible
  con   : no global view, can loop forever, hard to audit
  use   : research/debate, code-review pairs


  SUPERVISOR              HIERARCHICAL                 PEER-TO-PEER

      [S]                     [Top]                      [A]---[B]
    /  |  \                  /     \                      | \ / |
  [A] [B] [C]            [S1]       [S2]                  |  X  |
   \   |   /             /  \     

☝ The trade is **control vs hops**. A supervisor sees every result, so you get one place to log, retry, and enforce policy — at the cost of two LLM calls per specialist (route in, decide again on return). Peer-to-peer halves the hops and loses the audit point, which in a regulated bank is usually disqualifying on its own. Hierarchical exists because one router's context window can't hold twenty agents' descriptions and still route well.

Default for banking work: **supervisor**. Reach for hierarchical only when the specialist list stops fitting in one routing prompt.

## 2. Message passing vs shared state

The deeper fork, and the one interviewers push on. Do agents exchange **messages** (each agent's context = what it was sent), or do they all read and write one **shared state** object?

In [2]:
COMPARISON = [
    ("Context each agent sees", "only the message it received", "the whole state object"),
    ("Coupling",                "loose — agents know the schema", "tight — agents know the state keys"),
    ("Token cost",              "low: send only what's needed", "high: state grows, everyone reads it"),
    ("Debuggability",           "trace = ordered message log", "snapshot of state at each step"),
    ("Parallelism",             "natural (queue + N workers)", "needs reducers to merge writes"),
    ("Frameworks",              "CrewAI, AutoGen, A2A", "LangGraph StateGraph"),
]
print(f"{'':26}| {'MESSAGE PASSING':32}| SHARED STATE")
print("-" * 96)
for aspect, msg, shared in COMPARISON:
    print(f"{aspect:26}| {msg:32}| {shared}")

                          | MESSAGE PASSING                 | SHARED STATE
------------------------------------------------------------------------------------------------
Context each agent sees   | only the message it received    | the whole state object
Coupling                  | loose — agents know the schema  | tight — agents know the state keys
Token cost                | low: send only what's needed    | high: state grows, everyone reads it
Debuggability             | trace = ordered message log     | snapshot of state at each step
Parallelism               | natural (queue + N workers)     | needs reducers to merge writes
Frameworks                | CrewAI, AutoGen, A2A            | LangGraph StateGraph


☝ LangGraph is a **shared state** framework: nodes receive the whole `State` and return partial updates merged by reducers (Day 02's `Annotated[list, operator.add]` is exactly this). CrewAI and AutoGen are **message passing**: a task's output becomes the next task's context.

Neither is "correct". Shared state wins when agents genuinely need each other's intermediate work (a fraud verdict depends on both the transaction history *and* the device signals). Message passing wins when the pipeline is a chain and each stage only needs its predecessor's output — you pay far fewer tokens, because you're not re-sending the world to every agent.

## 3. The same task, both ways

A loan pre-check: **retrieve** the applicant → **score** risk → **decide**. Build it as a shared-state LangGraph, then as a message-passing chain, and compare what each agent sees.

In [3]:
from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, START, END

APPLICANTS = {"A-1001": {"name": "ACME Ltd", "income": 480_000, "arrears": 0, "sector": "retail"}}

class LoanState(TypedDict):
    applicant_id: str
    record: dict
    risk: float
    decision: str
    trace: Annotated[list[str], operator.add]      # reducer: every node appends

def retrieve(s: LoanState) -> dict:
    rec = APPLICANTS[s["applicant_id"]]
    return {"record": rec, "trace": [f"retriever saw keys={sorted(s.keys())}"]}

def score(s: LoanState) -> dict:
    rec = s["record"]                               # reads another agent's output directly
    risk = 0.1 if rec["arrears"] == 0 and rec["income"] > 250_000 else 0.7
    return {"risk": risk, "trace": [f"scorer saw record + keys={sorted(s.keys())}"]}

def decide(s: LoanState) -> dict:
    d = "APPROVE" if s["risk"] < 0.3 else "REFER"
    return {"decision": d, "trace": [f"decider saw risk={s['risk']}"]}

g = StateGraph(LoanState)
for name, fn in [("retrieve", retrieve), ("score", score), ("decide", decide)]:
    g.add_node(name, fn)
g.add_edge(START, "retrieve"); g.add_edge("retrieve", "score")
g.add_edge("score", "decide"); g.add_edge("decide", END)
graph = g.compile()

out = graph.invoke({"applicant_id": "A-1001", "trace": []})
print("SHARED STATE (LangGraph)")
for line in out["trace"]:
    print("  ", line)
print("   decision:", out["decision"])

SHARED STATE (LangGraph)
   retriever saw keys=['applicant_id', 'trace']
   scorer saw record + keys=['applicant_id', 'record', 'trace']
   decider saw risk=0.1
   decision: APPROVE


In [4]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Message:
    """Message passing: an agent's entire context is what it was sent."""
    sender: str
    to: str
    content: dict

def retriever_agent(msg: Message) -> Message:
    rec = APPLICANTS[msg.content["applicant_id"]]
    return Message("retriever", "scorer", {"record": rec})

def scorer_agent(msg: Message) -> Message:
    rec = msg.content["record"]
    risk = 0.1 if rec["arrears"] == 0 and rec["income"] > 250_000 else 0.7
    return Message("scorer", "decider", {"risk": risk})       # record NOT forwarded

def decider_agent(msg: Message) -> Message:
    d = "APPROVE" if msg.content["risk"] < 0.3 else "REFER"
    return Message("decider", "user", {"decision": d})

print("MESSAGE PASSING (CrewAI/AutoGen style)")
m = Message("user", "retriever", {"applicant_id": "A-1001"})
for agent in (retriever_agent, scorer_agent, decider_agent):
    m = agent(m)
    print(f"   {m.sender:10} -> {m.to:8} carries {list(m.content)}")
print("   decision:", m.content["decision"])

MESSAGE PASSING (CrewAI/AutoGen style)
   retriever  -> scorer   carries ['record']
   scorer     -> decider  carries ['risk']
   decider    -> user     carries ['decision']
   decision: APPROVE


☝ Same answer, different physics. In the graph, `score` read `s["record"]` without anyone forwarding it, and by `decide` the state held *everything* — convenient, and quietly expensive once `record` is a 40-page credit file that every node's prompt now includes. In the message chain, `decider` never sees the applicant record at all; it gets `{"risk": 0.1}`, which is all it needs — cheaper, but if the decider later *does* need the sector, someone must change the scorer to forward it.

That's the trade in one line: **shared state couples agents through a schema; message passing couples them through a contract.** LangGraph gives you the first, CrewAI the second, and Day 32/33 build one of each.

## 4. Barclays mappings

Two real pipelines, two different answers. This is the part of the interview where naming the topology *and the reason* is the whole point.

In [5]:
LOAN_PROCESSING = {
    "topology": "SUPERVISOR (shared state)",
    "why": "regulated decision — one router owns the audit trail and the fallback path",
    "agents": [
        ("supervisor",      "routes, owns the final decision, logs every hop"),
        ("doc-extractor",   "OCR + field extraction from statements"),
        ("credit-retriever","pulls bureau data + internal history"),
        ("risk-scorer",     "affordability + PD model call"),
        ("compliance",      "affordability rules, vulnerable-customer flags"),
    ],
    "human_gate": "supervisor interrupts before any REFUSE or >£250k APPROVE",
}

FRAUD_DETECTION = {
    "topology": "PIPELINE + PEER ESCALATION (message passing)",
    "why": "latency budget is ~200ms — every extra routing hop is a real cost",
    "agents": [
        ("stream-filter",  "rules pass, drops 99% of traffic, no LLM"),
        ("enricher",       "device + geo + velocity features"),
        ("analyst",        "LLM reasons over enriched signal, hands off to investigator"),
        ("investigator",   "peer: pulls history, drafts SAR narrative"),
    ],
    "human_gate": "investigator output -> human analyst queue, never auto-blocks",
}

for case, cfg in [("LOAN PROCESSING", LOAN_PROCESSING), ("FRAUD DETECTION", FRAUD_DETECTION)]:
    print(f"{case}  ->  {cfg['topology']}")
    print(f"  why: {cfg['why']}")
    for name, role in cfg["agents"]:
        print(f"    - {name:16} {role}")
    print(f"  human gate: {cfg['human_gate']}\n")

LOAN PROCESSING  ->  SUPERVISOR (shared state)
  why: regulated decision — one router owns the audit trail and the fallback path
    - supervisor       routes, owns the final decision, logs every hop
    - doc-extractor    OCR + field extraction from statements
    - credit-retriever pulls bureau data + internal history
    - risk-scorer      affordability + PD model call
    - compliance       affordability rules, vulnerable-customer flags
  human gate: supervisor interrupts before any REFUSE or >£250k APPROVE

FRAUD DETECTION  ->  PIPELINE + PEER ESCALATION (message passing)
  why: latency budget is ~200ms — every extra routing hop is a real cost
    - stream-filter    rules pass, drops 99% of traffic, no LLM
    - enricher         device + geo + velocity features
    - analyst          LLM reasons over enriched signal, hands off to investigator
    - investigator     peer: pulls history, drafts SAR narrative
  human gate: investigator output -> human analyst queue, never auto-blocks

☝ Read the two `why` lines together — they're the same reasoning applied to different constraints. Loan processing is slow (minutes are fine) and heavily regulated, so you buy auditability with extra hops: supervisor topology. Fraud detection has a hard latency budget, so you spend hops only where they earn their keep: a *non-LLM* rules filter kills 99% of traffic before any agent wakes up, and the surviving 1% flows through a chain.

Both keep a **human gate**, and neither lets an agent take the final action. In banking that isn't a design preference — it's the thing that makes the system deployable at all.

## 5. What a handoff actually costs

The most common multi-agent mistake is using five agents where one would do. Price it.

In [6]:
def cost_of(n_specialists: int, supervisor: bool, tokens_per_call: int = 1500) -> dict:
    """LLM calls for one request: supervisor routes in and re-decides on each return."""
    calls = n_specialists * (2 if supervisor else 1) + (1 if supervisor else 0)
    return {
        "agents": n_specialists,
        "llm_calls": calls,
        "tokens": calls * tokens_per_call,
        "latency_s": round(calls * 1.2, 1),          # ~1.2s per call
    }

print(f"{'design':34}| {'calls':>6}| {'tokens':>7}| {'latency':>8}")
print("-" * 62)
designs = [
    ("1 agent, 4 tools (no handoffs)",      cost_of(1, supervisor=False)),
    ("supervisor + 2 specialists",          cost_of(2, supervisor=True)),
    ("supervisor + 4 specialists",          cost_of(4, supervisor=True)),
    ("hierarchical, 2 teams x 3 (approx.)", cost_of(8, supervisor=True)),
]
for label, c in designs:
    print(f"{label:34}| {c['llm_calls']:>6}| {c['tokens']:>7}| {c['latency_s']:>7}s")

print("\nsplit into agents when ANY of these is true:")
for rule in [
    "the tool list no longer fits one prompt without degrading tool choice (~>10 tools)",
    "two sub-tasks need genuinely different system prompts or models (Haiku vs Sonnet)",
    "a sub-task needs its own access boundary (compliance must not write to the ledger)",
    "a sub-task is independently reusable across products",
]:
    print("  -", rule)

design                            |  calls|  tokens|  latency
--------------------------------------------------------------
1 agent, 4 tools (no handoffs)    |      1|    1500|     1.2s
supervisor + 2 specialists        |      5|    7500|     6.0s
supervisor + 4 specialists        |      9|   13500|    10.8s
hierarchical, 2 teams x 3 (approx.)|     17|   25500|    20.4s

split into agents when ANY of these is true:
  - the tool list no longer fits one prompt without degrading tool choice (~>10 tools)
  - two sub-tasks need genuinely different system prompts or models (Haiku vs Sonnet)
  - a sub-task needs its own access boundary (compliance must not write to the ledger)
  - a sub-task is independently reusable across products


☝ Four specialists behind a supervisor cost **9 LLM calls and ~11 seconds** where a single tool-using agent costs 1 call and ~1.2s. That is a 9x bill and a 9x latency budget, and it buys you nothing unless one of the four rules above applies. The honest answer to "should this be multi-agent?" is usually **no** — and being the person in the room who says that, with the arithmetic, reads as seniority rather than reluctance.

The rules that *do* justify a split are the ones you'll defend at AVP level: prompt/tool overload, different models per sub-task, and **access boundaries** — the compliance agent physically cannot write to the ledger, which is an architectural control an auditor can verify.

## 6. Recap — topology is a decision you must defend

| Question | Answer |
|---|---|
| Default topology for banking? | **Supervisor** — one audit point, one fallback path |
| When hierarchical? | specialists no longer fit one routing prompt (~>8) |
| When peer-to-peer? | latency-critical or exploratory; rarely in regulated flows |
| Shared state vs messages? | shared = agents need each other's work (LangGraph); messages = chain, cheaper (CrewAI/AutoGen) |
| When to add an agent? | tool overload, different model/prompt, access boundary, reuse |
| Cost of a handoff | ~2 LLM calls + ~2.4s + full context re-read |

Multi-agent design is mostly **subtraction**: start with one agent and four tools, and split only where a rule forces it. When you do split, put a supervisor in front so there's one place that logs, retries, and gates on a human.

Tomorrow: **CrewAI** — role-based agents, the message-passing end of this spectrum. Day 33: the **LangGraph supervisor** — the shared-state end. You'll have built both and can say, from experience, which one you'd take to production and why.